# Golo v2: cifrar un lote y abrirlo

Formato v2 `global(cdn_pass + cdn)`: UN solo PRBX con el **pass global**
sobre pass pegada + contenido: `PRBX(maestro, [LOTE][u32 len][pass][datos])`.
Con el global salen el pass del lote (texto) y el archivo. v1 sigue abriendo.

In [ ]:
import os, sys
if not os.path.isdir('/tmp/golo/python/toolsec.py'):
    !git clone --depth 1 https://github.com/elmasber-ma/golo.git /tmp/golo
sys.path.insert(0, '/tmp/golo/python')
from deps import ensure
print('motor:', ensure())

In [ ]:
from toolsec import Vault

MAESTRO = 'global-demo-123'
LOTE = 'cdn-lote-7'
with open('/tmp/test.txt', 'wb') as f:
    f.write(b'hola lote v2 ' * 1000)

v = Vault(MAESTRO)
out = v.encrypt_lote_file('/tmp/test.txt', MAESTRO, LOTE, '/tmp/test.prbx')
import os as _os
print('OK lote ->', out, _os.path.getsize(out), 'bytes')

In [ ]:
from toolsec import Vault

v = Vault('global-demo-123')
r = v.decrypt_lote_file('/tmp/test.prbx', 'global-demo-123', '/tmp/test.dec')
assert r is not None, 'no abrio'
pass_lote, path = r
print('pass del lote:', pass_lote)
a = open('/tmp/test.txt', 'rb').read()
b = open(path, 'rb').read()
print('contenido igual:', a == b)

## Abrirlo en Dart (en este mismo Colab)

Instala Dart SDK, abre con Dart el lote que cifró Python y viceversa.
Mismo envelope: lo que cifra uno lo abre el otro.

In [ ]:
!which dart || (sudo apt-get update -qq && sudo apt-get install -y -qq apt-transport-https wget gnupg) 2>&1 | tail -n 1
!which dart || (wget -qO- https://dl-ssl.google.com/linux/linux_signing_key.pub | sudo gpg --dearmor -o /usr/share/keyrings/dart.gpg && echo 'deb [signed-by=/usr/share/keyrings/dart.gpg arch=amd64] https://storage.googleapis.com/download.dartlang.org/linux/debian stable main' | sudo tee /etc/apt/sources.list.d/dart_stable.list && sudo apt-get update -qq && sudo apt-get install -y -qq dart) 2>&1 | tail -n 1
!dart --version
!cd /tmp/golo/cifre/dart && dart pub get 2>&1 | tail -n 2

In [ ]:
# Python cifro -> Dart abre (pass en texto + contenido)
!cd /tmp/golo/cifre/dart && dart run vault.dart dec-lote global-demo-123 /tmp/test.prbx /tmp/test_dart.dec && cmp /tmp/test.txt /tmp/test_dart.dec && echo "DART ABRE V2 OK"

In [ ]:
# Dart cifra -> Python abre
!cd /tmp/golo/cifre/dart && dart run vault.dart enc-lote global-demo-123 cdn-lote-7 /tmp/test.txt /tmp/test_dart.prbx
from toolsec import Vault
r = Vault('x').decrypt_lote_file('/tmp/test_dart.prbx', 'global-demo-123', '/tmp/test_py.dec')
print('pass:', r[0])
a = open('/tmp/test.txt','rb').read()
b = open(r[1],'rb').read()
print('cruce DART->PY igual:', a == b)